In [ ]:

# ============================================================
# SECTION 1: IMPORT LIBRARIES AND CONFIGURATION
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ------------------------------------------------------------
# Device configuration
# ------------------------------------------------------------

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("Selected device:", DEVICE)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "preprocessed"

MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results" / "training"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("\nProject root:", PROJECT_ROOT)
print("Dataset directory:", DATA_DIR)
print("Model directory:", MODEL_DIR)
print("Results directory:", RESULTS_DIR)

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

IMAGE_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 30
LEARNING_RATE = 1e-4

NUM_WORKERS = 0

print("\nTraining configuration:")
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)


In [ ]:

# ============================================================
# SECTION 2: CUSTOM DATASET LOADER
# ============================================================

class BUSIDataset(Dataset):

    def __init__(self, root_dir, split):

        self.root_dir = Path(root_dir)
        self.split = split

        self.image_dir = self.root_dir / split / "images"
        self.mask_dir = self.root_dir / split / "masks"

        if not self.image_dir.exists():
            raise FileNotFoundError(
                f"Image directory not found: {self.image_dir}"
            )

        if not self.mask_dir.exists():
            raise FileNotFoundError(
                f"Mask directory not found: {self.mask_dir}"
            )

        self.image_files = sorted(
            self.image_dir.glob("*.png")
        )

        self.mask_files = sorted(
            self.mask_dir.glob("*.png")
        )

        if len(self.image_files) != len(self.mask_files):
            raise ValueError(
                f"Image-mask count mismatch in {split}: "
                f"{len(self.image_files)} images, "
                f"{len(self.mask_files)} masks"
            )

        # Ensure corresponding filenames match
        image_names = [f.name for f in self.image_files]
        mask_names = [f.name for f in self.mask_files]

        if image_names != mask_names:
            raise ValueError(
                f"Image-mask filenames do not match in {split}"
            )

        print(f"{split.capitalize()} dataset: {len(self.image_files)} images")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):

        image_path = self.image_files[idx]
        mask_path = self.mask_files[idx]

        # Load grayscale image
        image = Image.open(image_path).convert("L")

        # Load grayscale mask
        mask = Image.open(mask_path).convert("L")

        # Convert to NumPy arrays
        image = np.array(image, dtype=np.float32) / 255.0
        mask = np.array(mask, dtype=np.float32)

        # Binarize segmentation mask
        mask = (mask > 0).astype(np.float32)

        # Convert to PyTorch tensors
        image = torch.from_numpy(image).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset = BUSIDataset(DATA_DIR, "train")
val_dataset = BUSIDataset(DATA_DIR, "validation")
test_dataset = BUSIDataset(DATA_DIR, "test")


# ------------------------------------------------------------
# Create DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)


# ------------------------------------------------------------
# Verify dataset
# ------------------------------------------------------------

print("\nDataset verification:")

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

images, masks = next(iter(train_loader))

print("\nBatch verification:")
print("Image tensor shape:", images.shape)
print("Mask tensor shape:", masks.shape)

print("Image value range:",
      images.min().item(), "to", images.max().item())

print("Mask unique values:", torch.unique(masks))

print("Image dtype:", images.dtype)
print("Mask dtype:", masks.dtype)


In [ ]:

# ============================================================
# SECTION 3: LIGHTWEIGHT U-NET ARCHITECTURE
# ============================================================

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):

    def __init__(self, in_channels=1, out_channels=1):

        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)

        # Bottleneck
        self.bottleneck = DoubleConv(128, 256)

        # Pooling
        self.pool = nn.MaxPool2d(kernel_size=2)

        # Decoder
        self.up3 = nn.ConvTranspose2d(
            256, 128, kernel_size=2, stride=2
        )
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(
            128, 64, kernel_size=2, stride=2
        )
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(
            64, 32, kernel_size=2, stride=2
        )
        self.dec1 = DoubleConv(64, 32)

        # Output layer
        self.output = nn.Conv2d(
            32, out_channels, kernel_size=1
        )

    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        # Bottleneck
        b = self.bottleneck(
            self.pool(e3)
        )

        # Decoder + skip connections
        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        # Raw logits (no sigmoid here)
        return self.output(d1)


# ------------------------------------------------------------
# Initialize model
# ------------------------------------------------------------

model = UNet(
    in_channels=1,
    out_channels=1
).to(DEVICE)


# ------------------------------------------------------------
# Model verification
# ------------------------------------------------------------

total_params = sum(
    p.numel() for p in model.parameters()
)

print("U-Net initialized successfully.")

print("Device:", DEVICE)

print("Total trainable parameters:",
      f"{total_params:,}")

# Test forward pass
sample_images, sample_masks = next(iter(train_loader))

sample_images = sample_images.to(DEVICE)

with torch.no_grad():
    sample_output = model(sample_images)

print("\nForward pass verification:")
print("Input shape:", sample_images.shape)
print("Output shape:", sample_output.shape)

print("Output device:", sample_output.device)


In [ ]:

# ============================================================
# SECTION 4: LOSS FUNCTIONS AND EVALUATION METRICS
# ============================================================

# ------------------------------------------------------------
# 1. Dice Loss
# ------------------------------------------------------------

def dice_loss(logits, targets, smooth=1e-6):

    probabilities = torch.sigmoid(logits)

    probabilities = probabilities.contiguous().view(
        probabilities.shape[0], -1
    )

    targets = targets.contiguous().view(
        targets.shape[0], -1
    )

    intersection = (probabilities * targets).sum(dim=1)

    denominator = (
        probabilities.sum(dim=1) +
        targets.sum(dim=1)
    )

    dice = (
        2.0 * intersection + smooth
    ) / (
        denominator + smooth
    )

    return 1.0 - dice.mean()


# ------------------------------------------------------------
# 2. Combined BCE + Dice Loss
# ------------------------------------------------------------

bce_loss_fn = nn.BCEWithLogitsLoss()


def combined_loss(logits, targets):

    bce = bce_loss_fn(logits, targets)

    dice = dice_loss(logits, targets)

    return bce + dice


# ------------------------------------------------------------
# 3. Segmentation metrics
# ------------------------------------------------------------

def calculate_metrics(logits, targets, threshold=0.5):

    predictions = (
        torch.sigmoid(logits) > threshold
    ).float()

    predictions = predictions.view(
        predictions.shape[0], -1
    )

    targets = targets.view(
        targets.shape[0], -1
    )

    intersection = (predictions * targets).sum(dim=1)

    pred_sum = predictions.sum(dim=1)

    target_sum = targets.sum(dim=1)

    union = pred_sum + target_sum - intersection

    # Dice
    dice = (
        2 * intersection + 1e-6
    ) / (
        pred_sum + target_sum + 1e-6
    )

    # IoU
    iou = (
        intersection + 1e-6
    ) / (
        union + 1e-6
    )

    # Pixel accuracy
    correct = (predictions == targets).float().sum(dim=1)

    pixel_accuracy = correct / targets.shape[1]

    return {
        "dice": dice.mean().item(),
        "iou": iou.mean().item(),
        "pixel_accuracy": pixel_accuracy.mean().item()
    }


# ------------------------------------------------------------
# 4. Test loss and metrics
# ------------------------------------------------------------

sample_images, sample_masks = next(iter(train_loader))

sample_images = sample_images.to(DEVICE)
sample_masks = sample_masks.to(DEVICE)

with torch.no_grad():

    sample_logits = model(sample_images)

    loss_value = combined_loss(
        sample_logits,
        sample_masks
    )

    metrics = calculate_metrics(
        sample_logits,
        sample_masks
    )

print("Loss and metric functions verified.")

print("\nCombined loss:", loss_value.item())

print("\nSample metrics:")

for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")


In [ ]:

# ============================================================
# SECTION 5: BASELINE U-NET TRAINING
# ============================================================

import copy
import time

# ------------------------------------------------------------
# Optimizer and scheduler
# ------------------------------------------------------------

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=4
)

# ------------------------------------------------------------
# Training and validation functions
# ------------------------------------------------------------

def train_one_epoch(model, loader, optimizer):

    model.train()

    running_loss = 0.0
    total_samples = 0

    for images, masks in loader:

        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        optimizer.zero_grad()

        logits = model(images)

        loss = combined_loss(logits, masks)

        loss.backward()

        optimizer.step()

        batch_size = images.size(0)

        running_loss += loss.item() * batch_size
        total_samples += batch_size

    return running_loss / total_samples


def validate_one_epoch(model, loader):

    model.eval()

    running_loss = 0.0
    total_samples = 0

    all_dice = []
    all_iou = []
    all_accuracy = []

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(DEVICE)
            masks = masks.to(DEVICE)

            logits = model(images)

            loss = combined_loss(logits, masks)

            batch_size = images.size(0)

            running_loss += loss.item() * batch_size
            total_samples += batch_size

            metrics = calculate_metrics(logits, masks)

            all_dice.append(metrics["dice"])
            all_iou.append(metrics["iou"])
            all_accuracy.append(metrics["pixel_accuracy"])

    return {
        "loss": running_loss / total_samples,
        "dice": np.mean(all_dice),
        "iou": np.mean(all_iou),
        "pixel_accuracy": np.mean(all_accuracy)
    }


# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------

history = []

best_val_dice = -1.0

best_model_state = None

best_epoch = 0

checkpoint_path = MODEL_DIR / "baseline_unet_best.pth"

print("Starting baseline U-Net training...\n")

for epoch in range(EPOCHS):

    start_time = time.time()

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer
    )

    val_metrics = validate_one_epoch(
        model,
        val_loader
    )

    scheduler.step(val_metrics["dice"])

    current_lr = optimizer.param_groups[0]["lr"]

    epoch_time = time.time() - start_time

    # Record history
    epoch_record = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_dice": val_metrics["dice"],
        "val_iou": val_metrics["iou"],
        "val_pixel_accuracy": val_metrics["pixel_accuracy"],
        "learning_rate": current_lr
    }

    history.append(epoch_record)

    # Save best model
    if val_metrics["dice"] > best_val_dice:

        best_val_dice = val_metrics["dice"]

        best_epoch = epoch + 1

        best_model_state = copy.deepcopy(model.state_dict())

        torch.save(
            {
                "epoch": best_epoch,
                "model_state_dict": best_model_state,
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_dice": best_val_dice,
                "config": {
                    "image_size": IMAGE_SIZE,
                    "batch_size": BATCH_SIZE,
                    "learning_rate": LEARNING_RATE,
                    "seed": SEED
                }
            },
            checkpoint_path
        )

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_metrics['loss']:.4f} "
        f"| Val Dice: {val_metrics['dice']:.4f} "
        f"| Val IoU: {val_metrics['iou']:.4f} "
        f"| LR: {current_lr:.6f} "
        f"| Time: {epoch_time:.1f}s"
    )


# ------------------------------------------------------------
# Save training history
# ------------------------------------------------------------

history_df = pd.DataFrame(history)

history_path = RESULTS_DIR / "baseline_training_history.csv"

history_df.to_csv(history_path, index=False)

# Restore best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)

print("\nTraining completed.")

print("Best epoch:", best_epoch)

print("Best validation Dice:", round(best_val_dice, 4))

print("Best model saved:", checkpoint_path)

print("Training history saved:", history_path)


In [ ]:

# ============================================================
# SECTION 6: TRAINING CURVE VISUALIZATION
# ============================================================

history_df = pd.read_csv(
    RESULTS_DIR / "baseline_training_history.csv"
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs = history_df["epoch"]

# Training vs validation loss
axes[0].plot(
    epochs, history_df["train_loss"],
    label="Training Loss"
)

axes[0].plot(
    epochs, history_df["val_loss"],
    label="Validation Loss"
)

axes[0].set_title("Training and Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Validation Dice
axes[1].plot(
    epochs, history_df["val_dice"],
    marker="o",
    markersize=3
)

axes[1].set_title("Validation Dice Score")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice")
axes[1].grid(alpha=0.3)

# Validation IoU
axes[2].plot(
    epochs, history_df["val_iou"],
    marker="o",
    markersize=3
)

axes[2].set_title("Validation IoU Score")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("IoU")
axes[2].grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "baseline_training_curves.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Training curves saved successfully.")


In [ ]:

# ============================================================
# SECTION 7: LOAD BEST CHECKPOINT AND TEST EVALUATION
# ============================================================

# Load best checkpoint
checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("Best checkpoint loaded.")
print("Best epoch:", checkpoint["epoch"])
print("Validation Dice:", checkpoint["best_val_dice"])


# ------------------------------------------------------------
# Evaluate individual test images
# ------------------------------------------------------------

test_results = []

THRESHOLD = 0.5

with torch.no_grad():

    for batch_idx, (images, masks) in enumerate(test_loader):

        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        logits = model(images)

        predictions = (
            torch.sigmoid(logits) > THRESHOLD
        ).float()

        for i in range(images.shape[0]):

            pred = predictions[i].view(-1)
            target = masks[i].view(-1)

            intersection = (pred * target).sum().item()

            pred_pixels = pred.sum().item()
            target_pixels = target.sum().item()

            is_normal = target_pixels == 0

            dice = (
                2 * intersection / (pred_pixels + target_pixels)
                if (pred_pixels + target_pixels) > 0
                else 1.0
            )

            iou = (
                intersection / (pred_pixels + target_pixels - intersection)
                if (pred_pixels + target_pixels - intersection) > 0
                else 1.0
            )

            # Pixel-level metrics
            tp = intersection
            fp = pred_pixels - intersection
            fn = target_pixels - intersection

            test_results.append({
                "image_index": len(test_results),
                "is_normal": is_normal,
                "ground_truth_pixels": target_pixels,
                "predicted_pixels": pred_pixels,
                "dice": dice,
                "iou": iou,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "false_positive_normal": (
                    is_normal and pred_pixels > 0
                )
            })


# ------------------------------------------------------------
# Convert results into DataFrame
# ------------------------------------------------------------

test_df = pd.DataFrame(test_results)

lesion_df = test_df[~test_df["is_normal"]].copy()

normal_df = test_df[test_df["is_normal"]].copy()


# ------------------------------------------------------------
# Calculate aggregate metrics
# ------------------------------------------------------------

total_tp = test_df["tp"].sum()
total_fp = test_df["fp"].sum()
total_fn = test_df["fn"].sum()

precision = (
    total_tp / (total_tp + total_fp)
    if total_tp + total_fp > 0 else 0
)

sensitivity = (
    total_tp / (total_tp + total_fn)
    if total_tp + total_fn > 0 else 0
)

lesion_dice = lesion_df["dice"].mean()
lesion_iou = lesion_df["iou"].mean()

normal_false_positives = int(
    normal_df["false_positive_normal"].sum()
)

normal_false_positive_rate = (
    normal_false_positives / len(normal_df)
    if len(normal_df) > 0 else 0
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n========== BASELINE TEST RESULTS ==========")

print("\nLesion segmentation:")
print("Lesion images:", len(lesion_df))
print(f"Mean Dice: {lesion_dice:.4f}")
print(f"Mean IoU: {lesion_iou:.4f}")

print("\nPixel-level metrics:")
print(f"Precision: {precision:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")

print("\nNormal image evaluation:")
print("Normal images:", len(normal_df))
print(
    "False-positive normal images:",
    normal_false_positives
)
print(
    f"Normal false-positive rate: "
    f"{normal_false_positive_rate:.4f}"
)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

test_results_path = (
    RESULTS_DIR / "baseline_test_results.csv"
)

test_df.to_csv(test_results_path, index=False)

print("\nResults saved:", test_results_path)
